# Setup verification
Brief code snippets to check that the various components of the robot setup are working correctly, are connected and so on. 

In [1]:
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"


## AL5D robot

### In Linux

* The power source should be plugged in and robot turned on
* This should create a serial device: /dev/ttyUSB0 (or sometimes USB1)

```sudo chmod o+rw /dev/ttyUSB0``` 

* As of July 2025, this is specified in al5d/pulse_controller_00.yaml, so you need to change it there to the current port

### In Windows


### In the end:

If everything is successful, this should wake up the robot, perform a short move, than turn it out

In [2]:
from robot.al5d_position_controller import PositionController

exp_robot_controller = Config().get_experiment("robot_al5d", "position_controller_00")

# Linux
# rob = PositionController("/dev/ttyUSB0") # USB1 or USB0 
# rob = PositionController(Config().values["robot"]["usb_port"]) # USB1 or USB0 
# rob = PositionController(exp_robot_controller["usb_port"])
rob = PositionController(exp_robot_controller)
# Windows (can be other COM#)
# rob = AL5D_PositionController("COM4")


***ExpRun**: Loading pointer config file:
	/home/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	/home/lboloni/Insync/lotzi.boloni@gmail.com/Google Drive/LotziStudy/Code/PackageTracking/BerryPicker/settings/settings-tredy2.yaml
***ExpRun**: Using torch device: cuda
***ExpRun**: Configuration for exp/run: robot_al5d/position_controller_00 successfully loaded
***ExpRun**: Configuration for exp/run: robot_al5d/pulse_controller_00 successfully loaded
***ExpRun**: Configuration for exp/run: robot_al5d/angle_controller_00 successfully loaded
{'height': 5.0, 'distance': 5.0, 'heading': 0.0, 'wrist_angle': -45.0, 'wrist_rotation': 75.0, 'gripper': 100}


In [3]:
pos = rob.get_position()
pos["height"] = pos["height"] + 1
rob.move(pos)

Shut down the robot: can be also used to terminate if other code dies.

In [4]:
rob.stop_robot()

***al5d_position_controller: Initiating the stopping of the robot
***al5d_position_controller: Robot stopped


# The camera

```
pip install opencv-python
pip install opencv-contrib-python
```

Capture device 0 is usually the webcam, the other ones seem to be 2... 

On Linux, one can check the devices as 

```
/dev/video*
```

In [5]:
import numpy as np
import cv2

In [ ]:
from pathlib import Path
from pprint import pprint
import re


def video_index_from_device(device_path):
    match = re.search(r"video(\d+)$", str(Path(device_path).resolve()))
    return int(match.group(1)) if match else None


def short_camera_name(stable_path, fallback_index):
    stem = stable_path.name
    if stem.startswith("usb-"):
        stem = stem[4:]
    stem = re.sub(r"-video-index\d+$", "", stem)
    stem = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_").lower()
    return stem or f"camera_{fallback_index}"


stable_camera_paths = []
for device_dir in (Path("/dev/v4l/by-id"), Path("/dev/v4l/by-path")):
    if device_dir.exists():
        stable_camera_paths.extend(sorted(device_dir.glob("*-video-index*")))

seen_devices = set()
cameras = {}
for stable_path in stable_camera_paths:
    device_path = stable_path.resolve()
    if device_path in seen_devices:
        continue
    seen_devices.add(device_path)

    opencv_id = video_index_from_device(device_path)
    cap = cv2.VideoCapture(str(stable_path), cv2.CAP_V4L2)
    opened = cap.isOpened()
    ret, frame = cap.read() if opened else (False, None)

    camera_name = short_camera_name(stable_path, len(cameras))
    if camera_name in cameras:
        camera_name = f"{camera_name}_{opencv_id}"

    fourcc_int = int(cap.get(cv2.CAP_PROP_FOURCC)) if opened else 0
    fourcc = "".join(chr((fourcc_int >> 8 * i) & 0xFF) for i in range(4)).strip("\x00")

    cameras[camera_name] = {
        "type": "v4l2_camera",
        "opencv_id": opencv_id,
        "hardware_path": str(stable_path),
        "device_path": str(device_path),
        "opened": opened,
        "read_ok": ret,
        "frame_shape": tuple(frame.shape) if ret else None,
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) if opened else None,
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if opened else None,
        "fps": cap.get(cv2.CAP_PROP_FPS) if opened else None,
        "fourcc": fourcc,
        "backend": cap.getBackendName() if opened else None,
    }
    cap.release()

if not cameras:
    print("No stable camera hardware paths found under /dev/v4l/by-id or /dev/v4l/by-path.")
else:
    print("Stable camera hardware-path inventory:")
    pprint(cameras)


In [15]:
# identifying which cameras work
for i in range(10):
    cap = cv2.VideoCapture(i)
    ret, frame = cap.read()
    if ret:
        print(f"Camera {i} is working")
    else:
        print(f"Camera {i} not working")
    cap.release()

Camera 0 is working
Camera 1 is working
Camera 2 is working
Camera 3 not working
Camera 4 not working
Camera 5 not working
Camera 6 not working
Camera 7 not working
Camera 8 not working
Camera 9 not working


[ WARN:0@366.076] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video3): can't open camera by index
[ERROR:0@366.078] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@366.078] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video4): can't open camera by index
[ERROR:0@366.080] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@366.080] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video5): can't open camera by index
[ERROR:0@366.081] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@366.081] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video6): can't open camera by index
[ERROR:0@366.083] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
[ WARN:0@366.083] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video7): can't open camera by index
[ERROR:0@366.084] global obsensor_uvc_stream_channel.cpp:158 get

## Testing a single camera

Specify the camera as camera_number. Values should be one of those that show up in Linux as /dev/video*. 

If the camera with the camera_number works well, it will show up in a window. Exit the window by pressing "q" while the window being the currently active. 

In [14]:
#camera_number = Config().values["robot"]["active_camera_list"][0]
# Experiment
camera_number = 0
print(f"Camera={camera_number}")
cap = cv2.VideoCapture(camera_number)
while(True):
    ret, frame = cap.read()
    if not ret:
        print("Capture frame returned False, exiting the loop")
        break
    # imgresized = cv2.resize(frame, (640, 480))
    imgresized = frame
    cv2.imshow(f'Camera {camera_number} - q to exit', frame)
    ret = cv2.waitKey(1)
    if ret != -1:
        ret2 = ret & 0xFF
        if ret2 == ord('q'):
            break
# When everything done, release the capture
cap.release()
cv2.destroyAllWindows()

Camera=0


## Testing multiple cameras

In all the setups tried until now (May 2024), simultaneously reading from more than 2 camera slows things down to be unusable. Apparently, some USB related problem. 

In [1]:
DIM = (256, 256)


# create the capture devices
capture_devs = {}
# camera0 - webcam on the computer
# camera2 - right mounted
# camera3 - the free floating one
# camera4 - the center mounted one 
# cameras = [0, 2, 3, 4]
# cameras = Config().values["robot"]["active_camera_list"]
cameras = [0, 1]
for i in cameras:
    cap = cv2.VideoCapture(i) 
    # set the resolution it works with 160x120
    # cap.set(3,160)
    # cap.set(4,120)
    # set the resolution it works with 160x120
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 256)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 256)
    cap.set(cv2.CAP_PROP_FPS, 10)
    #cap.set(3,160)
    #cap.set(4,120)

    if cap is None or not cap.isOpened():
        print(f"Warning: unable to open video source: {i}")
    else:
        capture_devs[f"dev{i}"] = cap
        print(f"cap{i} works")

while True:
    images = []
    for index in capture_devs.keys():
        cap = capture_devs[index]
        ret, frame = cap.read()
        if ret:
            # FIXME: probably we want to crop as well??? 
            imgresized = cv2.resize(frame, DIM)
            images.append(imgresized)
    concatenated_image = cv2.hconcat(images)
    cv2.imshow('Press q to exit', concatenated_image)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting the acquisition loop as a result of a q key")
        break

# When everything done, release the capture
cap.release()
cv2.destroyAllWindows()

NameError: name 'cv2' is not defined

# The game controller
* Making sure that the game controller is working
* The version which is working with the Voyee 360 is the approxeng.input 2.5
* There was some kind of version interaction problem with version 2.6, it is unclear whether this is specific to my combinations in the hf workspace

* https://approxeng.github.io/approxeng.input/commandline.html 

```
pip install approxeng.input-2.5
```

Also note the approxeng.input-2.5 only works with Python 3.10, not higher. 

# These are the buttons that we can read out
['square']
['triangle']
['circle']
['cross']
['l1']
['r1']
['ls']
['rs']
['ls']
['rs']
['dleft']
['dright']
['dup']
['ddown']

In [ ]:
from approxeng.input.selectbinder import ControllerResource, ControllerNotFoundError
import time

try:
    with ControllerResource() as joystick:
        print('Found a joystick and connected. Press select / back to exit loop')
        print(joystick.controls)
        while joystick.connected:
            presses = joystick.check_presses()
            if len(presses.buttons) > 0:
                print(presses.names)
                print(joystick)
                print(joystick["dleft"]) # if held, returns the seconds since held
            if "select" in presses.names:
                break
except ControllerNotFoundError as e:
    print("Controller not found")
    print(e)
print("Bye")